# Đánh giá YOLO11m fine-tuned trên VisDrone DET test-dev

**Chỉ đánh giá một checkpoint `best.pt`, không huấn luyện lại.**

- Đầu vào 1: Kaggle Dataset `tiendq/visdrone-mot-det-converted` (đã giải nén, có `uav_tracking_mot_det/data/...`).
- Đầu vào 2: Kaggle Dataset đã lưu checkpoint `T1_yolo11m_640_det/.../weights/best.pt`.
- Chạy **đủ 1.610 ảnh DET test-dev**, không lấy mẫu.
- Dùng 10 lớp VisDrone nguyên bản.
- Đầu ra: `test_metrics.json`, `per_class_ap_test.csv`, `test_summary.csv`, confusion matrices, cấu hình đánh giá và ZIP.

Trên Kaggle: **Add Input cả hai dataset**, bật GPU, bật Internet nếu cần cài `ultralytics`.

> `test-dev` có nhãn để tính mAP. `test-challenge` không được dùng ở đây.

In [1]:
%pip -q install "ultralytics==8.4.197" pandas pyyaml

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import json, os, sys, platform, shutil, zipfile
import numpy as np
import pandas as pd
import yaml
import torch
import ultralytics
from ultralytics import YOLO

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
# Cấu hình
DATASET_ROOT = Path('/kaggle/input/datasets/tiendq/visdrone-mot-det-converted/uav_tracking_mot_det')
BEST_PT = Path('/kaggle/input/datasets/tiendq/result-train-det-yolo11m-80e/T1_yolo11m_640_det_artifacts/T1_yolo11m_640_det/weights/best.pt')    
IMGSZ = 640
BATCH = 16         
DEVICE = 0
WORKERS = 2
MAX_DET = 1000

STAGE = WORK/'visdrone_det_test_eval'
OUTPUT = WORK/'T1_yolo11m_640_det_test_eval'

if not (DATASET_ROOT/'data'/'processed'/'yolo_det').is_dir():
    roots = [p for p in INPUT.glob('*/uav_tracking_mot_det')
             if (p/'data'/'processed'/'yolo_det').is_dir()]
    if len(roots) != 1:
        raise FileNotFoundError(f'Cần đúng một dataset VisDrone đã chuyển đổi. Tìm thấy: {roots}')
    DATASET_ROOT = roots[0]

if BEST_PT is None:
    patterns = [
        '*/T1_yolo11m_640_det_artifacts/T1_yolo11m_640_det/weights/best.pt',
        '*/T1_yolo11m_640_det/weights/best.pt',
        '*/weights/best.pt',
    ]
    candidates = [p for pattern in patterns for p in INPUT.glob(pattern)]
    candidates += list(WORK.glob('yolo11_runs/T1_yolo11m_640_det/weights/best.pt'))
    candidates = list(dict.fromkeys(p for p in candidates if p.is_file()))
    if len(candidates) != 1:
        # Chỉ fallback scan trong input dataset liên quan đến YOLO/checkpoint
        extra = []
        for d in INPUT.iterdir():
            if d.is_dir() and any(s in d.name.lower() for s in ('yolo','checkpoint','artifact','weight')):
                extra.extend(d.rglob('best.pt'))
        candidates = list(dict.fromkeys(candidates + [p for p in extra if p.is_file()]))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Không xác định duy nhất best.pt: {candidates}. Hãy đặt BEST_PT thủ công.')
    BEST_PT = candidates[0]
else:
    BEST_PT = Path(BEST_PT)
assert BEST_PT.is_file(), f'Không tìm thấy best.pt: {BEST_PT}'

print('Dataset:', DATASET_ROOT)
print('Checkpoint:', BEST_PT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Ultralytics:', ultralytics.__version__)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Dataset: /kaggle/input/datasets/tiendq/visdrone-mot-det-converted/uav_tracking_mot_det
Checkpoint: /kaggle/input/datasets/tiendq/result-train-det-yolo11m-80e/T1_yolo11m_640_det_artifacts/T1_yolo11m_640_det/weights/best.pt
GPU: Tesla T4
Ultralytics: 8.4.197


In [3]:
# Khôi phục riêng DET test-dev. Không copy ảnh; chỉ tạo symlink vào /kaggle/working.
source = DATASET_ROOT/'data'/'processed'/'yolo_det'
coco_path = DATASET_ROOT/'data'/'processed'/'coco_det'/'test-dev.json'
labels_src = source/'test-dev'/'labels'
assert coco_path.is_file() and labels_src.is_dir()

coco = json.loads(coco_path.read_text(encoding='utf-8'))
image_map = {}
for im in coco['images']:
    assert im.get('source') == 'det', f'Không phải ảnh DET: {im}'
    label_image_name = 'det_' + Path(im['file_name']).name
    raw_image = DATASET_ROOT/'data'/'raw'/im['file_name']
    if label_image_name in image_map:
        raise ValueError(f'Trùng tên ảnh: {label_image_name}')
    image_map[label_image_name] = raw_image

if STAGE.exists():
    shutil.rmtree(STAGE)
images_dir = STAGE/'test-dev'/'images'
labels_dir = STAGE/'test-dev'/'labels'
images_dir.mkdir(parents=True)
labels_dir.mkdir(parents=True)
label_files = sorted(labels_src.glob('*.txt'))
assert len(label_files) == len(image_map), (len(label_files), len(image_map))

missing = []
for label in label_files:
    image_name = label.with_suffix('.jpg').name
    raw_image = image_map.get(image_name)
    if raw_image is None or not raw_image.is_file():
        missing.append((image_name, str(raw_image)))
        continue
    (images_dir/image_name).symlink_to(raw_image)
    (labels_dir/label.name).symlink_to(label)
assert not missing, f'Thiếu {len(missing)} ảnh hoặc mapping; ví dụ: {missing[:3]}'

src_yaml = yaml.safe_load((source/'visdrone.yaml').read_text(encoding='utf-8'))
assert len(src_yaml['names']) == 10, 'Dataset phải có 10 lớp VisDrone'
test_yaml = {
    'path': str(STAGE),
    'train': 'test-dev/images', # chỉ để YAML đầy đủ, KHÔNG train
    'val': 'test-dev/images',   # chỉ để YAML đầy đủ
    'test': 'test-dev/images',  # được chọn bởi split="test"
    'names': src_yaml['names'],
}
TEST_YAML = STAGE/'visdrone_test.yaml'
TEST_YAML.write_text(yaml.safe_dump(test_yaml,sort_keys=False,allow_unicode=True),encoding='utf-8')
print('Số ảnh DET test-dev:',len(label_files))
assert len(label_files) == 1610, f'Mong đợi 1610 ảnh DET test-dev, thực tế {len(label_files)}'
print('YAML:',TEST_YAML)

Số ảnh DET test-dev: 1610
YAML: /kaggle/working/visdrone_det_test_eval/visdrone_test.yaml


In [4]:
# Inference + metric chính thức của Ultralytics trên toàn bộ test-dev.
# mAP50-95, mAP50, mAP75, Precision, Recall; plots và confusion matrix.
model = YOLO(str(BEST_PT))
assert len(model.names) == 10, f'Checkpoint không phải YOLO 10 lớp: {model.names}'
metrics = model.val(
    data=str(TEST_YAML),
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    conf=0.001,
    iou=0.7,
    max_det=MAX_DET,
    plots=True,
    project=str(WORK),
    name=OUTPUT.name,
    exist_ok=True,
    verbose=True,
)
print(f'\nmAP50-95: {metrics.box.map:.4f}')
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP75:    {metrics.box.map75:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

Ultralytics 8.4.197 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11m summary (fused): 125 layers, 20,037,742 parameters, 0 gradients, 67.8 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 49.6±8.3 MB/s, size: 172.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/working/visdrone_det_test_eval/test-dev/labels... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 364.8it/s 4.4s
val: New cache created: /kaggle/working/visdrone_det_test_eval/test-dev/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 101/101 2.9it/s 34.5s
                   all       1610      75102        0.5      0.402      0.377      0.219
            pedestrian       1197      21006      0.513      0.351      0.337      0.138
                people        797       6376    

In [ ]:
# Xuất metrics; per-class chỉ từ những lớp thực sự được đánh giá.
OUTPUT.mkdir(parents=True,exist_ok=True)
names = {int(k): str(v) for k,v in model.names.items()}
summary = {
    'checkpoint': str(BEST_PT),
    'dataset': str(DATASET_ROOT),
    'split': 'DET test-dev',
    'n_images': len(label_files),
    'n_classes': 10,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'conf': 0.001,
    'iou_nms': 0.7,
    'max_det': MAX_DET,
    'mAP50_95': float(metrics.box.map),
    'mAP50': float(metrics.box.map50),
    'mAP75': float(metrics.box.map75),
    'precision_mean': float(metrics.box.mp),
    'recall_mean': float(metrics.box.mr),
    'speed_ms_per_image': {str(k):float(v) for k,v in metrics.speed.items()},
    'ultralytics_version':ultralytics.__version__,
    'torch_version':torch.__version__,
    'python_version':platform.python_version(),
    'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
}
(OUTPUT/'test_metrics.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
pd.DataFrame([summary]).to_csv(OUTPUT/'test_summary.csv',index=False)

ap_ids=np.asarray(metrics.box.ap_class_index,dtype=int).reshape(-1)
per_class=[]
for row, class_id in enumerate(ap_ids):
    per_class.append({
        'class_id':int(class_id), 'class':names[int(class_id)],
        'precision':float(metrics.box.p[row]),
        'recall':float(metrics.box.r[row]),
        'AP50':float(metrics.box.ap50[row]),
        'AP50_95':float(metrics.box.ap[row]),
    })
per_class_df=pd.DataFrame(per_class).sort_values('class_id')
per_class_df.to_csv(OUTPUT/'per_class_ap_test.csv',index=False)
(OUTPUT/'test_eval_config.yaml').write_text(yaml.safe_dump({
    'checkpoint':str(BEST_PT),'data':str(TEST_YAML),'split':'test',
    'imgsz':IMGSZ,'batch':BATCH,'conf':0.001,'iou':0.7,'max_det':MAX_DET,
},sort_keys=False),encoding='utf-8')
print('\nMETRICS:')
display(pd.DataFrame([summary])[['mAP50_95','mAP50','mAP75','precision_mean','recall_mean']])
print('\nAP PER CLASS:')
display(per_class_df)
print('Lưu tại:',OUTPUT)


METRICS:


,mAP50_95,mAP50,mAP75,precision_mean,recall_mean
0,0.218603,0.37714,0.222165,0.499974,0.402409



AP PER CLASS:


,class_id,class,precision,recall,AP50,AP50_95
0,0,pedestrian,0.513363,0.350662,0.337341,0.138100
1,1,people,0.529762,0.162014,0.178367,0.060567
2,2,bicycle,0.287274,0.179724,0.142360,0.059469
3,3,car,0.719955,0.773207,0.763610,0.489221
4,4,van,0.485300,0.464986,0.423112,0.283962
5,5,truck,0.524658,0.492666,0.474530,0.313399
6,6,tricycle,0.309382,0.375472,0.248644,0.129168
7,7,awning-tricycle,0.407685,0.235392,0.223724,0.127795
8,8,bus,0.735354,0.561224,0.608928,0.433075
9,9,motor,0.487010,0.428743,0.370780,0.151270


Lưu tại: /kaggle/working/T1_yolo11m_640_det_test_eval


In [6]:
# Gói artifacts đánh giá, không đóng gói dataset hay checkpoint cũ.
ZIP_PATH = WORK/'T1_yolo11m_640_det_test_eval_artifacts.zip'
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
    for file in OUTPUT.rglob('*'):
        if file.is_file():
            zf.write(file,file.relative_to(WORK))
print('Đã tạo:',ZIP_PATH,'(',round(ZIP_PATH.stat().st_size/1024**2,2),'MB)')
print('Các file:',*[str(p.relative_to(OUTPUT)) for p in sorted(OUTPUT.rglob('*')) if p.is_file()],sep='\n- ')

Đã tạo: /kaggle/working/T1_yolo11m_640_det_test_eval_artifacts.zip ( 4.76 MB)
Các file:
- BoxF1_curve.png
- BoxPR_curve.png
- BoxP_curve.png
- BoxR_curve.png
- confusion_matrix.png
- confusion_matrix_normalized.png
- per_class_ap_test.csv
- test_eval_config.yaml
- test_metrics.json
- test_summary.csv
- val_batch0_labels.jpg
- val_batch0_pred.jpg
- val_batch1_labels.jpg
- val_batch1_pred.jpg
- val_batch2_labels.jpg
- val_batch2_pred.jpg
